In [5]:
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import re

def plot_fig1_from_harvest():
    # 你的数据目录 (万点长征结果)
    DATA_DIR = "riemann_10k_harvest"
    
    print(f">>> 🚀 启动数据挖掘机，目标目录: {DATA_DIR}")
    
    # 1. 获取所有 .npy 文件
    # 匹配格式: harvest_seg_0_k_8.5000.npy
    files = glob.glob(os.path.join(DATA_DIR, "harvest_seg_*.npy"))
    
    if not files:
        print(f"❌ 错误：在 {DATA_DIR} 下没找到文件！")
        print("请确认你已经运行了‘万点长征’代码，并且文件夹名字正确。")
        return

    print(f">>> 扫描到 {len(files)} 个数据文件，开始提取...")

    n_list = []
    k_list = []

    # 2. 遍历文件提取 (n, k)
    # 因为你的生成代码逻辑是: seg_idx -> mid_n -> k
    # 我们直接反推: mid_n = seg_idx * 100 + 50
    for f_path in files:
        try:
            basename = os.path.basename(f_path)
            
            # 使用正则提取 seg_idx 和 k
            # 文件名示例: harvest_seg_0_k_8.1234.npy
            match = re.search(r'harvest_seg_(\d+)_k_([\d\.]+)\.npy', basename)
            
            if match:
                seg_idx = int(match.group(1))
                k_val = float(match.group(2))
                
                # 核心：根据你的生成代码反推 n
                # start_n = seg_idx * 100
                # end_n = start_n + 100
                # mid_n = (start_n + end_n) / 2
                mid_n = seg_idx * 100 + 50
                
                if mid_n > 1: # 过滤掉非法的 n
                    n_list.append(mid_n)
                    k_list.append(k_val)
            
        except Exception as e:
            print(f"⚠️ 跳过文件 {basename}: {e}")

    # 3. 校验数据
    if len(n_list) == 0:
        print("❌ 提取失败：没有解析到有效数据。请检查文件名格式！")
        return

    # 排序
    n_arr = np.array(n_list)
    k_arr = np.array(k_list)
    
    idx = np.argsort(n_arr)
    n_arr = n_arr[idx]
    k_arr = k_arr[idx]

    print(f">>> ✅ 成功提取 {len(n_arr)} 个数据点！(n范围: {min(n_arr)} - {max(n_arr)})")

    # --- 4. 绘图 (PRL 风格) ---
    plt.figure(figsize=(12, 7), dpi=300)

    # A. 理论预测线 (The Red River)
    # 范围覆盖数据点
    n_pred = np.logspace(np.log10(max(2, min(n_arr))), np.log10(max(n_arr)*1.2), 1000)
    k_theory = 4.7 + 10.13 / np.log(n_pred)

    plt.semilogx(n_pred, k_theory, 'r-', lw=3, alpha=0.9, 
                 label=r'Theoretical Flow: $k(n) = 4.7 + \frac{10.13}{\ln n}$')

    # B. 实验数据点 (Cyan Dots)
    plt.semilogx(n_arr, k_arr, 'o', c='cyan', 
                 markeredgecolor='#008080', markeredgewidth=0.5, 
                 markersize=6, alpha=0.8, 
                 label=f'Harvested Data (Segments, N={len(n_arr)})')

    # C. 临界吸引子
    plt.axhline(4.7, color='green', ls='--', lw=1.5, label=r'Critical Attractor $k_c \approx 4.7$')

    # D. 装饰
    plt.title('Figure 1: Renormalization Group Flow Trajectory (Harvest Data)', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Riemann Zero Index $n$ (Log Scale)', fontsize=12)
    plt.ylabel('Coupling Constant $k$', fontsize=12)
    plt.legend(loc='upper right', frameon=True, framealpha=0.95)
    plt.grid(True, which="major", ls="-", alpha=0.2)
    plt.grid(True, which="minor", ls=":", alpha=0.1)
    
    # 标注
    k_final = 4.7 + 10.13 / np.log(10000)
    plt.text(10000, k_final + 0.3, f'Target\nn=10k\nk={k_final:.2f}', color='red', ha='center', fontsize=10)

    plt.xlim(max(1, min(n_arr)*0.8), 20000)
    plt.ylim(4.5, max(k_arr)*1.1)
    plt.tight_layout()

    # 保存
    save_path = "Figure_1_From_Harvest.png"
    plt.savefig(save_path)
    print(f">>> 🏆 图表已生成: {save_path}")
    plt.show()

if __name__ == "__main__":
    plot_fig1_from_harvest()

>>> 🚀 启动数据挖掘机，目标目录: riemann_10k_harvest
❌ 错误：在 riemann_10k_harvest 下没找到文件！
请确认你已经运行了‘万点长征’代码，并且文件夹名字正确。
